# AraForge — Notebook 2: Metadata & Phonemization (REVISED)
*Fixes applied:* (1) full **ECAPA-TDNN helper functions implemented** (were missing/commented, so the notebook could not run); (2) **corrected `IPA_RULES`** (Arabic ف/د restored, Egyptian `Al-`=/e/, Khaliji=/i/); (3) `Al-` article now **anchored to word start** (fixes قال→/gl/ mis-parse) and bare-alif `/aː/ recovered; (4) **dialect-aware `للـ`**; (5) the pipeline now **saves `Ara170_Gold_Standard.csv`** that Notebook 3 consumes (previously computed and discarded).

# 1. Environment Setup & Dependencies

In [ ]:
!pip install -q speechbrain torchaudio pandas numpy openpyxl
import os, re, random, warnings
import pandas as pd, numpy as np, torch, torchaudio
from speechbrain.pretrained import EncoderClassifier
from google.colab import drive
warnings.filterwarnings("ignore")
drive.mount('/content/drive', force_remount=True)
print("Environment ready.")

# 2. Global Configuration & Toggles

In [ ]:
DETECT_EMOTIONS    = False
SIMILARITY_THRESHOLD = 0.82       # paper threshold (median cosine)
SAMPLE_DURATION_SEC  = 20
NUM_SAMPLES          = 10

DB_ROOT = '/content/drive/MyDrive/Ara170_Database'; os.makedirs(DB_ROOT, exist_ok=True)
CSV_PATH        = os.path.join(DB_ROOT, 'speaker_metadata.csv')
EMBEDDINGS_PATH = os.path.join(DB_ROOT, 'speaker_embeddings.npy')
print(f"Speaker DB: {DB_ROOT} | threshold={SIMILARITY_THRESHOLD}")

# 3. Phase 1: Foreign-Language Tagging
*Wraps Latin/English runs in `<eng>…</eng>` so the Arabic G2P engine skips them. (Pure digit-only runs are left untagged to avoid wrapping Arabic-adjacent numerals.)*

In [ ]:
def tag_foreign_text(text):
    if pd.isna(text): return text
    text = str(text)
    # one or more Latin words (optionally space-separated); requires at least one a-zA-Z
    pattern = r'([A-Za-z][A-Za-z0-9]*(?:\s+[A-Za-z][A-Za-z0-9]*)*)'
    return re.sub(pattern, r'<eng>\1</eng>', text)

print(tag_foreign_text("خرج الboy من الفصل sad جدا"))

# 4. Phase 2: Speaker Verification (ECAPA-TDNN) — full implementation
*All helpers that were previously commented out are implemented here. ECAPA expects 16 kHz mono, so audio is resampled internally even though the corpus master is 22.05 kHz/stereo.*

In [ ]:
from speechbrain.inference.interfaces import foreign_class

print("Loading ECAPA-TDNN (spkrec-ecapa-voxceleb) ...")
CLASSIFIER = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb")
print("Loaded.")

# ---- persistent speaker database (globals) ----
METADATA_DF = pd.DataFrame(columns=['speaker_id','Common_Dialect','gender','age','num_projects'])
SPEAKER_EMBEDDINGS = {}   # speaker_id -> centroid np.ndarray
ECAPA_SR = 16000

def load_speaker_database():
    global METADATA_DF, SPEAKER_EMBEDDINGS
    if os.path.exists(CSV_PATH):        METADATA_DF = pd.read_csv(CSV_PATH)
    if os.path.exists(EMBEDDINGS_PATH): SPEAKER_EMBEDDINGS = np.load(EMBEDDINGS_PATH, allow_pickle=True).item()
    print(f"Loaded {len(METADATA_DF)} enrolled speaker(s).")

def save_speaker_database():
    METADATA_DF.to_csv(CSV_PATH, index=False)
    np.save(EMBEDDINGS_PATH, SPEAKER_EMBEDDINGS)

def load_and_preprocess_audio(path, target_sr=ECAPA_SR):
    signal, sr = torchaudio.load(path)
    if signal.shape[0] > 1: signal = signal.mean(dim=0, keepdim=True)   # stereo->mono
    if sr != target_sr:     signal = torchaudio.transforms.Resample(sr, target_sr)(signal)
    return signal

def extract_random_segment(waveform, target_duration_sec=SAMPLE_DURATION_SEC, sr=ECAPA_SR):
    total = waveform.shape[1]; want = int(target_duration_sec * sr)
    if total <= want: return waveform
    start = random.randint(0, total - want)
    return waveform[:, start:start + want]

def extract_embedding(segment):
    with torch.no_grad():
        emb = CLASSIFIER.encode_batch(segment)
    return emb.squeeze().detach().cpu().numpy()

def calculate_similarity(a, b):
    a = np.asarray(a).flatten(); b = np.asarray(b).flatten()
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

def find_speaker(new_embedding):
    best_id, best = None, -1.0
    for spk, cen in SPEAKER_EMBEDDINGS.items():
        s = calculate_similarity(new_embedding, cen)
        if s > best: best, best_id = s, spk
    return best_id, (best if best_id is not None else None)

def create_centroid_embedding(full_waveform, n=NUM_SAMPLES):
    embs = []
    for _ in range(n):
        seg = extract_random_segment(full_waveform)
        if seg is not None: embs.append(extract_embedding(seg))
    return np.mean(np.stack(embs), axis=0)

def handle_new_speaker(embedding, dialect, gender, age):
    global METADATA_DF, SPEAKER_EMBEDDINGS
    new_id = f"SPK_{random.randint(10000, 99999)}"
    while new_id in SPEAKER_EMBEDDINGS: new_id = f"SPK_{random.randint(10000, 99999)}"
    SPEAKER_EMBEDDINGS[new_id] = embedding
    METADATA_DF = pd.concat([METADATA_DF, pd.DataFrame([{
        'speaker_id': new_id, 'Common_Dialect': dialect, 'gender': gender, 'age': age, 'num_projects': 1
    }])], ignore_index=True)
    save_speaker_database()
    return new_id

if DETECT_EMOTIONS:
    print("Loading Wav2Vec2 emotion model (IEMOCAP) ...")
    EMOTION_CLASSIFIER = foreign_class(
        source="speechbrain/emotion-recognition-wav2vec2-IEMOCAP",
        pymodule_file="custom_interface.py",
        classname="CustomEncoderWav2vec2Classifier")
    print("Emotion model loaded.")

# 5. Phase 3: G2P Rule Engine — EXPERT-REVISED, synced to `IPA_Phonemes.xlsx`\n*Rules loaded from the (revised) spreadsheet. Linguistic corrections grounded in Arabic phonology: **ل is a sun letter** (الليل→/allayl/), **hamza = /ʔ/** (no spurious /æ/), **bare alif = /aː/** medial & **/ʔ/** word-initial, **fatha+alif merge** to one /aː/, and **shadda+vowel** consumed together. Short vowels via **diacritize-then-G2P** (`DIACRITIZE_HOOK`, CAMeL/Mishkal).*

In [ ]:
import json, re
# ============================================================
# Arabic G2P — rules loaded from IPA_Phonemes.xlsx (single source of truth),
# expert-revised: lam is a sun letter; hamza = /ʔ/; bare alif = /aː/ (word-initial /ʔ/);
# short vowels via diacritize-then-G2P (DIACRITIZE_HOOK).
# ============================================================
XLSX_PATH = "/content/drive/MyDrive/IPA_Phonemes.xlsx"   # <-- set to your (revised) xlsx

SUN_LETTERS  = set("تثدذرزسشصضطظلن")     # 14, INCLUDING lam
MOON_LETTERS = set("ءأإآبجحخعغفقكمهوي")

def _clean(x): return "" if x is None else str(x).replace("\t","").strip()
def _norm(p):
    p=_clean(p)
    if p=="" or p.lower() in ("silent","no vowel"): return ""
    if p.lower()=="consonant doubling": return "doubled"
    return p

def load_ipa_rules_from_xlsx(path):
    import openpyxl
    wb=openpyxl.load_workbook(path, data_only=True); ws=wb['IPA']
    DIAL_COL={"Egyptian":2,"Khaliji Arabic":6,"Levantine Arabic":10,"Modern Standard Arabic (MSA)":14}
    letters={d:{} for d in DIAL_COL}
    for r in ws.iter_rows(values_only=True,min_row=3,max_row=ws.max_row):
        g=_clean(r[1])
        if not g: continue
        for d,ci in DIAL_COL.items():
            ph=_norm(r[ci])
            if ph!="" and g not in letters[d]: letters[d][g]={"phoneme":ph}
    wsd=wb['Diacritic']; diac={}
    for r in wsd.iter_rows(values_only=True,min_row=2,max_row=9):
        k=_clean(r[0])
        if k: diac[k]=_norm(r[1])
    short={k:diac[k] for k in ("\u064e","\u064f","\u0650") if k in diac}
    tanwin={k:diac[k] for k in ("\u064b","\u064c","\u064d") if k in diac}
    other={"\u0652":diac.get("\u0652",""),"\u0651":"doubled"}
    wsm=wb['Sun and Moon Letters']
    rows=[[_clean(c) for c in row[:8]] for row in wsm.iter_rows(values_only=True,min_row=3,max_row=7)]
    moon=next(r for r in rows if r[0]=="Moon Letters"); sun=next(r for r in rows if r[0]=="Sun Letters")
    DCOL={"Egyptian":3,"Khaliji Arabic":4,"Levantine Arabic":5,"Modern Standard Arabic (MSA)":6}
    fa=lambda s:s.split(" or ")[0].split(" +")[0].strip()
    sun_set=set(sun[1].replace(",","").replace(" ",""))   # now includes ل
    R={"dialects":{}}
    for d in DIAL_COL:
        R["dialects"][d]={"letters":letters[d],
            "vowels":{"short":dict(short),"long":{"ا":"/aː/","و":"/uː/","ي":"/iː/"}},
            "diacritics":{**tanwin,**other},
            "phonological_rules":{
                "al_prefix_moon":{"letters":moon[1].replace("هـ","ه"),"ipa":fa(moon[DCOL[d]])},
                "al_prefix_sun":{"letters":"".join(sorted(sun_set)),"ipa":fa(sun[DCOL[d]])}},
            "lil_prefix":"/lel-/" if d=="Egyptian" else ("/lal-/" if "MSA" in d else "/lil-/")}
        R["dialects"][d]["letters"].setdefault("ا",{"phoneme":"/aː/"})
    return R

try:
    IPA_RULES = load_ipa_rules_from_xlsx(XLSX_PATH)
    print(f"[IPA_RULES] loaded from xlsx: {XLSX_PATH}")
except Exception as e:
    print(f"[IPA_RULES] xlsx unavailable ({e}); using embedded fallback.")
    IPA_RULES = json.loads(r'''{"dialects": {"Egyptian": {"letters": {"ء": {"phoneme": "/ʔ/"}, "أ": {"phoneme": "/ʔ/"}, "إ": {"phoneme": "/ʔi/"}, "آ": {"phoneme": "/ʔaː/"}, "ئ": {"phoneme": "/ʔ/"}, "ؤ": {"phoneme": "/ʔ/"}, "ب": {"phoneme": "/b/"}, "ت": {"phoneme": "/t/"}, "ث": {"phoneme": "/t/"}, "ج": {"phoneme": "/ɡ/"}, "ح": {"phoneme": "/ħ/"}, "خ": {"phoneme": "/x/"}, "د": {"phoneme": "/d/"}, "ذ": {"phoneme": "/d/"}, "ر": {"phoneme": "/ɾ/"}, "ز": {"phoneme": "/z/"}, "س": {"phoneme": "/s/"}, "ش": {"phoneme": "/ʃ/"}, "ص": {"phoneme": "/sˤ/"}, "ض": {"phoneme": "/dˤ/"}, "ط": {"phoneme": "/tˤ/"}, "ظ": {"phoneme": "/zˤ/"}, "ع": {"phoneme": "/ʕ/"}, "غ": {"phoneme": "/ɣ/"}, "ف": {"phoneme": "/f/"}, "ق": {"phoneme": "/ʔ/"}, "ك": {"phoneme": "/k/"}, "ل": {"phoneme": "/l/"}, "م": {"phoneme": "/m/"}, "ن": {"phoneme": "/n/"}, "ه": {"phoneme": "/h/"}, "و": {"phoneme": "/w/"}, "ي": {"phoneme": "/j/"}, "ة": {"phoneme": "/a/ or silent"}, "ا": {"phoneme": "/aː/"}}, "vowels": {"short": {"َ": "/a/", "ُ": "/u/", "ِ": "/i/"}, "long": {"ا": "/aː/", "و": "/uː/", "ي": "/iː/"}}, "diacritics": {"ً": "/an/", "ٌ": "/un/", "ٍ": "/in/", "ْ": "", "ّ": "doubled"}, "phonological_rules": {"al_prefix_moon": {"letters": "ا, ب, ج, ح, خ, ع, غ, ف, ق, ك, م, ه, و, ي", "ipa": "/el-/"}, "al_prefix_sun": {"letters": "تثدذرزسشصضطظلن", "ipa": "/e-/"}}, "lil_prefix": "/lel-/"}, "Khaliji Arabic": {"letters": {"ء": {"phoneme": "/ʔ/"}, "أ": {"phoneme": "/ʔ/"}, "إ": {"phoneme": "/ʔi/"}, "آ": {"phoneme": "/ʔaː/"}, "ئ": {"phoneme": "/ʔ/"}, "ؤ": {"phoneme": "/ʔ/"}, "ب": {"phoneme": "/b/"}, "ت": {"phoneme": "/t/"}, "ث": {"phoneme": "/θ/"}, "ج": {"phoneme": "/dʒ/"}, "ح": {"phoneme": "/ħ/"}, "خ": {"phoneme": "/x/"}, "د": {"phoneme": "/d/"}, "ذ": {"phoneme": "/ð/"}, "ر": {"phoneme": "/r/"}, "ز": {"phoneme": "/z/"}, "س": {"phoneme": "/s/"}, "ش": {"phoneme": "/ʃ/"}, "ص": {"phoneme": "/sˤ/"}, "ض": {"phoneme": "/dˤ/"}, "ط": {"phoneme": "/tˤ/"}, "ظ": {"phoneme": "/ðˤ/"}, "ع": {"phoneme": "/ʕ/"}, "غ": {"phoneme": "/ɣ/"}, "ف": {"phoneme": "/f/"}, "ق": {"phoneme": "/g/"}, "ك": {"phoneme": "/k/"}, "ل": {"phoneme": "/l/"}, "م": {"phoneme": "/m/"}, "ن": {"phoneme": "/n/"}, "ه": {"phoneme": "/h/"}, "و": {"phoneme": "/w/"}, "ي": {"phoneme": "/j/"}, "ة": {"phoneme": "/a/"}, "ا": {"phoneme": "/aː/"}}, "vowels": {"short": {"َ": "/a/", "ُ": "/u/", "ِ": "/i/"}, "long": {"ا": "/aː/", "و": "/uː/", "ي": "/iː/"}}, "diacritics": {"ً": "/an/", "ٌ": "/un/", "ٍ": "/in/", "ْ": "", "ّ": "doubled"}, "phonological_rules": {"al_prefix_moon": {"letters": "ا, ب, ج, ح, خ, ع, غ, ف, ق, ك, م, ه, و, ي", "ipa": "/ɪl-/"}, "al_prefix_sun": {"letters": "تثدذرزسشصضطظلن", "ipa": "/ɪ-/"}}, "lil_prefix": "/lil-/"}, "Levantine Arabic": {"letters": {"ء": {"phoneme": "/ʔ/"}, "أ": {"phoneme": "/ʔ/"}, "إ": {"phoneme": "/ʔi/"}, "آ": {"phoneme": "/ʔaː/"}, "ئ": {"phoneme": "/ʔ/"}, "ؤ": {"phoneme": "/ʔ/"}, "ب": {"phoneme": "/b/"}, "ت": {"phoneme": "/t/"}, "ث": {"phoneme": "/t/"}, "ج": {"phoneme": "/ʒ/"}, "ح": {"phoneme": "/ħ/"}, "خ": {"phoneme": "/x/"}, "د": {"phoneme": "/d/"}, "ذ": {"phoneme": "/d/"}, "ر": {"phoneme": "/r/"}, "ز": {"phoneme": "/z/"}, "س": {"phoneme": "/s/"}, "ش": {"phoneme": "/ʃ/"}, "ص": {"phoneme": "/sˤ/"}, "ض": {"phoneme": "/dˤ/"}, "ط": {"phoneme": "/tˤ/"}, "ظ": {"phoneme": "/zˤ/"}, "ع": {"phoneme": "/ʕ/"}, "غ": {"phoneme": "/ɣ/"}, "ف": {"phoneme": "/f/"}, "ق": {"phoneme": "/ʔ/"}, "ك": {"phoneme": "/k/"}, "ل": {"phoneme": "/l/"}, "م": {"phoneme": "/m/"}, "ن": {"phoneme": "/n/"}, "ه": {"phoneme": "/h/"}, "و": {"phoneme": "/w/"}, "ي": {"phoneme": "/j/"}, "ة": {"phoneme": "/a/"}, "ا": {"phoneme": "/aː/"}}, "vowels": {"short": {"َ": "/a/", "ُ": "/u/", "ِ": "/i/"}, "long": {"ا": "/aː/", "و": "/uː/", "ي": "/iː/"}}, "diacritics": {"ً": "/an/", "ٌ": "/un/", "ٍ": "/in/", "ْ": "", "ّ": "doubled"}, "phonological_rules": {"al_prefix_moon": {"letters": "ا, ب, ج, ح, خ, ع, غ, ف, ق, ك, م, ه, و, ي", "ipa": "/ɪl-/"}, "al_prefix_sun": {"letters": "تثدذرزسشصضطظلن", "ipa": "/ɪ-/"}}, "lil_prefix": "/lil-/"}, "Modern Standard Arabic (MSA)": {"letters": {"ء": {"phoneme": "/ʔ/"}, "أ": {"phoneme": "/ʔ/"}, "إ": {"phoneme": "/ʔi/"}, "آ": {"phoneme": "/ʔaː/"}, "ئ": {"phoneme": "/ʔ/"}, "ؤ": {"phoneme": "/ʔ/"}, "ب": {"phoneme": "/b/"}, "ت": {"phoneme": "/t/"}, "ث": {"phoneme": "/θ/"}, "ج": {"phoneme": "/dʒ/"}, "ح": {"phoneme": "/ħ/"}, "خ": {"phoneme": "/x/"}, "د": {"phoneme": "/d/"}, "ذ": {"phoneme": "/ð/"}, "ر": {"phoneme": "/r/"}, "ز": {"phoneme": "/z/"}, "س": {"phoneme": "/s/"}, "ش": {"phoneme": "/ʃ/"}, "ص": {"phoneme": "/sˤ/"}, "ض": {"phoneme": "/dˤ/"}, "ط": {"phoneme": "/tˤ/"}, "ظ": {"phoneme": "/ðˤ/"}, "ع": {"phoneme": "/ʕ/"}, "غ": {"phoneme": "/ɣ/"}, "ف": {"phoneme": "/f/"}, "ق": {"phoneme": "/q/"}, "ك": {"phoneme": "/k/"}, "ل": {"phoneme": "/l/"}, "م": {"phoneme": "/m/"}, "ن": {"phoneme": "/n/"}, "ه": {"phoneme": "/h/"}, "و": {"phoneme": "/w/"}, "ي": {"phoneme": "/j/"}, "ة": {"phoneme": "/a/"}, "ا": {"phoneme": "/aː/"}}, "vowels": {"short": {"َ": "/a/", "ُ": "/u/", "ِ": "/i/"}, "long": {"ا": "/aː/", "و": "/uː/", "ي": "/iː/"}}, "diacritics": {"ً": "/an/", "ٌ": "/un/", "ٍ": "/in/", "ْ": "", "ّ": "doubled"}, "phonological_rules": {"al_prefix_moon": {"letters": "ا, ب, ج, ح, خ, ع, غ, ف, ق, ك, م, ه, و, ي", "ipa": "/al-/"}, "al_prefix_sun": {"letters": "تثدذرزسشصضطظلن", "ipa": "/a-/"}}, "lil_prefix": "/lal-/"}}}''')

def DIACRITIZE_HOOK(text, enable=False):
    """Recover short vowels BEFORE G2P. Tries CAMeL Tools, then Mishkal; identity fallback."""
    if not enable: return text
    try:
        from camel_tools.disambig.mle import MLEDisambiguator
        mle=MLEDisambiguator.pretrained(); out=[]
        for dd in mle.disambiguate(text.split()):
            diac=dd.analyses[0].analysis.get('diac') if dd.analyses else None
            out.append(diac or dd.word)
        return " ".join(out)
    except Exception: pass
    try:
        from mishkal.tashkeel import TashkeelClass
        return TashkeelClass().tashkeel(text)
    except Exception: return text

def get_rules(dialect):
    if not isinstance(dialect,str): return IPA_RULES["dialects"]["Modern Standard Arabic (MSA)"]
    if "Egyptian" in dialect: return IPA_RULES["dialects"]["Egyptian"]
    if "Levantine" in dialect: return IPA_RULES["dialects"]["Levantine Arabic"]
    if "Khaliji" in dialect or "Gulf" in dialect: return IPA_RULES["dialects"]["Khaliji Arabic"]
    return IPA_RULES["dialects"]["Modern Standard Arabic (MSA)"]

def _consume_diacritics(word,j,n,diacritics,units):
    shadda=False; vowel=None
    while j<n and word[j] in diacritics:
        if word[j]=='\u0651': shadda=True
        else:
            v=diacritics[word[j]]
            if v not in ("","doubled"): vowel=v.strip('/')
        j+=1
    if shadda and units: units.append(units[-1])
    if vowel: units.append(vowel)
    return j

def transcribe_arabic(word,letters,diacritics,sun_letters,rules):
    units=[]; i=0; n=len(word)
    while i<n:
        ch=word[i]
        if i==0 and ch=='\u0644' and i+1<n and word[i+1]=='\u0644':
            units.append(rules.get("lil_prefix","/lil-/").strip('/')); i+=2; continue
        if i==0 and ch=='\u0627' and i+1<n and word[i+1]=='\u0644':
            if i+2<n and word[i+2] in sun_letters:
                pre=rules["phonological_rules"]["al_prefix_sun"]["ipa"].strip('/')
                sp=letters.get(word[i+2],{}).get("phoneme","").split(" or ")[0].strip('/')
                units.append(f"{pre}{sp}{sp}"); i+=3
                i=_consume_diacritics(word,i,n,diacritics,units); continue
            else:
                units.append(rules["phonological_rules"]["al_prefix_moon"]["ipa"].strip('/')); i+=2; continue
        if ch=='\u0627':
            if i==0: units.append("ʔ")
            elif units and units[-1]=="a": units[-1]="aː"
            else: units.append("aː")
            i=_consume_diacritics(word,i+1,n,diacritics,units); continue
        # taa marbuta: /a/ in pausal, but DO NOT double if preceded by fatha (last unit "a")
        if ch == '\u0629':                                  # ة
            if not (units and units[-1] == "a"):
                units.append("a")
            i += 1; continue
        if ch in letters:
            ph=letters[ch]["phoneme"]
            if " or " in ph: ph=ph.split(" or ")[0]
            if ph: units.append(ph.strip('/'))
            i=_consume_diacritics(word,i+1,n,diacritics,units); continue
        i+=1
    joined="".join(units)
    return f"/{joined}/" if joined else ""

def transcribe_protected(text,dialect,diacritize=False):
    if text is None or text=="" : return ""
    text=DIACRITIZE_HOOK(str(text),enable=diacritize)
    rules=get_rules(dialect); letters=rules["letters"]
    diacritics={**rules["vowels"]["short"],**rules["diacritics"]}
    spec=rules["phonological_rules"]["al_prefix_sun"].get("letters")
    sun=set(spec.replace(",","").replace(" ","")) if spec else SUN_LETTERS
    out=[]
    for chunk in re.split(r'(<eng>.*?</eng>)',str(text)):
        if chunk.startswith("<eng>") and chunk.endswith("</eng>"): out.append(chunk)
        elif chunk.strip():
            for w in chunk.strip().split():
                p=transcribe_arabic(w,letters,diacritics,sun,rules)
                if p: out.append(p)
    return " ".join(out)


# ---- Short-vowel recovery via diacritize-then-G2P ----
# Without a diacritizer, output is a consonant+long-vowel skeleton.
print("skeleton :", transcribe_protected("كتب مدرس", "Egyptian", diacritize=False))
# With diacritization ON (requires: pip install camel-tools  OR  mishkal), short vowels appear.
# Pre-diacritized text also works directly:
print("diacritized:", transcribe_protected("كَتَبَ مُدَرِّس", "Modern Standard Arabic (MSA)", diacritize=False))
# To enable automatic diacritization, install a diacritizer and call with diacritize=True:
# !pip install camel-tools && camel_data -i disambig-mle-calima-msa-r13
# print(transcribe_protected("كتب مدرس", "Modern Standard Arabic (MSA)", diacritize=True))

# Self-test (run to validate the rules)

In [ ]:
# ---- Self-test: validates the rules without the full dataset ----
def _check(word, dialect, contains=None, equals=None, absent=None, diacritize=False):
    out = transcribe_protected(word, dialect, diacritize=diacritize); ok=True
    if equals is not None and out!=equals: ok=False
    if contains is not None and contains not in out: ok=False
    if absent is not None and absent in out: ok=False
    print(f"  [{'PASS' if ok else 'FAIL'}] {word} [{dialect[:4]}] -> {out}")
    return ok
CASES = [
    ("الشمس","Egyptian",dict(contains="ʃʃ")), ("الليل","Egyptian",dict(contains="ll")),
    ("الرحمن","Modern Standard Arabic (MSA)",dict(contains="rr")),
    ("القمر","Egyptian",dict(contains="el-")), ("الشمس","Khaliji Arabic",dict(contains="ɪ-")),
    ("سماء","Egyptian",dict(contains="ʔ",absent="æ")), ("آمن","Modern Standard Arabic (MSA)",dict(contains="ʔaː")),
    ("اسم","Egyptian",dict(contains="ʔ",absent="aːsm")), ("قال","Khaliji Arabic",dict(equals="/gaːl/")),
    ("جميل","Egyptian",dict(contains="ɡ")), ("جميل","Levantine Arabic",dict(contains="ʒ")),
    ("قلب","Khaliji Arabic",dict(contains="g")), ("قلب","Modern Standard Arabic (MSA)",dict(contains="q")),
    ("سكّر","Egyptian",dict(contains="kk")),
    ("كَتَبَ","Egyptian",dict(equals="/kataba/")), ("كِتَاب","Modern Standard Arabic (MSA)",dict(contains="kitaːb")),
    ("مُدَرِّس","Modern Standard Arabic (MSA)",dict(contains="mudarris")),
    ("مَدْرَسَة","Modern Standard Arabic (MSA)",dict(equals="/madrasa/",absent="aa")),
    ("شَجَرَة","Modern Standard Arabic (MSA)",dict(equals="/ʃadʒara/")),
    ("خرج ال<eng>boy</eng>","Egyptian",dict(contains="<eng>boy</eng>")),
]
passed=sum(_check(w,d,**kw) for w,d,kw in CASES)
print(f"\n{passed}/{len(CASES)} self-tests passed.")

# 6. Phase 4: Execution Engine
*FIX: now writes `Ara170_Gold_Standard.csv` (pipe-separated) per project — the exact file Notebook 3 consumes.*

In [ ]:
def process_pipeline():
    parent_directory = input("Workspace directory (Notebook 1 output): ").strip()
    load_speaker_database()

    for sub_dir in os.listdir(parent_directory):
        sub_dir_path = os.path.join(parent_directory, sub_dir)
        if not os.path.isdir(sub_dir_path): continue
        main_folder_path = os.path.join(sub_dir_path, "Main")
        excel_path       = os.path.join(sub_dir_path, "final_transcriptions_mapping.xlsx")
        clean_audio_path = os.path.join(main_folder_path, "clean_master.wav")
        if not (os.path.exists(clean_audio_path) and os.path.exists(excel_path)): continue

        print(f"\n{'='*50}\nProcessing: {sub_dir}\n{'='*50}")
        df = pd.read_excel(excel_path)

        # 1) SPEAKER & DIALECT
        full_waveform = load_and_preprocess_audio(clean_audio_path)
        scores, best_id = [], None
        for _ in range(NUM_SAMPLES):
            seg = extract_random_segment(full_waveform)
            if seg is None: continue
            emb = extract_embedding(seg)
            cid, sc = find_speaker(emb)
            if sc is not None:
                scores.append(sc)
                if best_id is None or sc == max(scores): best_id = cid

        if scores and best_id is not None and np.median(scores) >= SIMILARITY_THRESHOLD:
            info = METADATA_DF[METADATA_DF['speaker_id'] == best_id].iloc[0]
            final_spk_id, final_dialect = best_id, info['Common_Dialect']
            print(f"Speaker matched -> {final_spk_id} ({final_dialect}), median={np.median(scores):.3f}")
        else:
            print("New speaker.")
            dialect = input("Dialect (Egyptian/Levantine Arabic/Khaliji Arabic/MSA): ").strip() or "Modern Standard Arabic (MSA)"
            gender  = input("Gender (M/F): ").strip() or "N/A"
            age     = input("Age: ").strip() or "N/A"
            centroid = create_centroid_embedding(full_waveform)
            final_spk_id = handle_new_speaker(centroid, dialect, gender, age)
            final_dialect = dialect

        df['Speaker_ID'] = final_spk_id
        df['Dialect']    = final_dialect

        # 2) FOREIGN TAGGING
        df['Transcript_Tagged'] = df['Transcript'].apply(tag_foreign_text)

        # 3) G2P
        df['Phonemes'] = df.apply(lambda r: transcribe_protected(r['Transcript_Tagged'], r['Dialect']), axis=1)

        # 4) EMOTION (optional)
        if DETECT_EMOTIONS:
            emotions, conf = [], []; parts_dir = os.path.join(sub_dir_path, "Parts")
            labels = ["happy","sad","angry","neutral"]
            for _, row in df.iterrows():
                ap = os.path.join(parts_dir, f"{row['Audio Clip Number']}.wav")
                if os.path.exists(ap):
                    sig, _ = torchaudio.load(ap)
                    out = EMOTION_CLASSIFIER.classify_batch(sig); p = out[1]
                    emotions.append(labels[p.argmax().item()]); conf.append(p.max().item())
                else:
                    emotions.append("neutral"); conf.append(1.0)
            df["Emotion"], df["Confidence Score"] = emotions, conf
        else:
            df["Emotion"], df["Confidence Score"] = "neutral", 1.0

        # 5) SAVE Gold Standard CSV (consumed by Notebook 3)  --- previously missing ---
        gold_path = os.path.join(sub_dir_path, "Ara170_Gold_Standard.csv")
        df.to_csv(gold_path, sep='|', encoding='utf-8', index=False)
        print(f"Saved -> {gold_path}  ({len(df)} rows)")

    save_speaker_database()
    print("\nNotebook 2 complete. Proceed to Notebook 3.")

process_pipeline()